# Gale-Shapley: Stable Matching Simulator

Gale-Shapley solves the stable matching problem: pair two groups so there is no pair who would both rather be matched with each other than with their assigned partners.

The algorithm began as a mathematical model of college admissions, then became part of real market design: medical residency matching, school choice, kidney exchange, and other allocation systems. It is tied to the 2012 Nobel Prize in Economics through Lloyd Shapley and Alvin Roth's work on stable allocations.

<details>
<summary>Big idea</summary>

One side proposes in preference order. The other side keeps its favorite proposal so far and rejects the rest. The process stops when no proposer is unmatched and still has someone left to ask.

</details>

## 1. The Mental Model

Stable matching uses two groups and preference lists:

- **Proposer**: an unmatched person who asks their next choice
- **Receiver**: a person who can tentatively hold one proposal
- **Preference list**: ranked choices from favorite to least favorite
- **Tentative match**: a match that can be replaced by a better proposal
- **Rejected proposer**: someone who becomes free and tries again
- **Blocking pair**: two people who prefer each other over their assigned matches

A matching is stable when there are no blocking pairs.

<details>
<summary>Hint: tentative does not mean final</summary>

A receiver can hold a proposal now and still switch later if a more preferred proposer arrives.

</details>

## 2. Build the Objects

Implementation plan:

1. `Person` stores a participant name.
2. `MatchingMarket` stores both groups and their preferences.
3. `ProposalStep` records each proposal, rejection, and tentative match.
4. `GaleShapleyRunner` owns the stable matching logic.
5. `MatchingReplay` prints the proposal history.

<details>
<summary>Implementation hint</summary>

Build a rank lookup for receivers. Lower rank means more preferred, so comparing two proposers becomes fast.

</details>

**Object model.** Define `Person`, the named objects used by the next examples.


In [ ]:
from collections import deque

from dataclasses import dataclass

@dataclass(frozen=True, order=True)
class Person:
    name: str

    def __str__(self) -> str:
        return self.name

    def __format__(self, spec: str) -> str:
        return format(self.name, spec)


**Object model.** Define `MatchingMarket`, the named objects used by the next examples.


In [ ]:
@dataclass
class MatchingMarket:
    proposers: list[Person]
    receivers: list[Person]
    proposer_preferences: dict[Person, list[Person]]
    receiver_preferences: dict[Person, list[Person]]

    def __post_init__(self) -> None:
        if len(self.proposers) != len(self.receivers):
            raise ValueError("This lesson version expects equal-sized groups.")

        self.receiver_ranks = {
            receiver: {proposer: rank for rank, proposer in enumerate(preferences)}
            for receiver, preferences in self.receiver_preferences.items()
        }

    def receiver_prefers(self, receiver: Person, new_proposer: Person, current_proposer: Person) -> bool:
        ranks = self.receiver_ranks[receiver]
        return ranks[new_proposer] < ranks[current_proposer]

    def describe(self) -> str:
        rows = ["Proposer preferences:"]
        for proposer in self.proposers:
            rows.append(f"  {proposer:>7}: {self._format_list(self.proposer_preferences[proposer])}")

        rows.append("Receiver preferences:")
        for receiver in self.receivers:
            rows.append(f"  {receiver:>7}: {self._format_list(self.receiver_preferences[receiver])}")

        return "\n".join(rows)

    def _format_list(self, people: list[Person]) -> str:
        return " > ".join(str(person) for person in people)


**Trace model.** Define `ProposalStep`, `MatchingResult`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass
class ProposalStep:
    round_number: int
    proposer: Person
    receiver: Person
    action: str
    matches_by_receiver: dict[Person, Person]
    free_proposers: list[Person]

@dataclass
class MatchingResult:
    matches_by_proposer: dict[Person, Person]
    steps: list[ProposalStep]
    blocking_pairs: list[tuple[Person, Person]]

    @property
    def is_stable(self) -> bool:
        return not self.blocking_pairs


**Algorithm engine.** Define `GaleShapleyRunner`, the class that runs the main simulation or algorithm.


In [ ]:
class GaleShapleyRunner:
    def __init__(self, market: MatchingMarket):
        self.market = market

    def run(self) -> MatchingResult:
        free_proposers = deque(self.market.proposers)
        next_choice_index = {proposer: 0 for proposer in self.market.proposers}
        matches_by_receiver: dict[Person, Person] = {}
        steps: list[ProposalStep] = []
        round_number = 0

        while free_proposers:
            proposer = free_proposers.popleft()
            preferences = self.market.proposer_preferences[proposer]

            if next_choice_index[proposer] >= len(preferences):
                continue

            receiver = preferences[next_choice_index[proposer]]
            next_choice_index[proposer] += 1
            current_match = matches_by_receiver.get(receiver)
            round_number += 1

            if current_match is None:
                matches_by_receiver[receiver] = proposer
                action = f"{receiver} is free, so {receiver} holds {proposer}."
            elif self.market.receiver_prefers(receiver, proposer, current_match):
                matches_by_receiver[receiver] = proposer
                if next_choice_index[current_match] < len(self.market.proposer_preferences[current_match]):
                    free_proposers.append(current_match)
                action = f"{receiver} switches from {current_match} to {proposer}."
            else:
                if next_choice_index[proposer] < len(preferences):
                    free_proposers.append(proposer)
                action = f"{receiver} rejects {proposer} and keeps {current_match}."

            steps.append(
                ProposalStep(
                    round_number=round_number,
                    proposer=proposer,
                    receiver=receiver,
                    action=action,
                    matches_by_receiver=matches_by_receiver.copy(),
                    free_proposers=list(free_proposers),
                )
            )

        matches_by_proposer = {proposer: receiver for receiver, proposer in matches_by_receiver.items()}
        blocking_pairs = self.find_blocking_pairs(matches_by_proposer)
        return MatchingResult(matches_by_proposer, steps, blocking_pairs)

    def find_blocking_pairs(self, matches_by_proposer: dict[Person, Person]) -> list[tuple[Person, Person]]:
        matches_by_receiver = {receiver: proposer for proposer, receiver in matches_by_proposer.items()}
        blocking_pairs = []

        for proposer in self.market.proposers:
            current_receiver = matches_by_proposer.get(proposer)
            preferences = self.market.proposer_preferences[proposer]
            preferred_receivers = preferences if current_receiver is None else preferences[:preferences.index(current_receiver)]

            for receiver in preferred_receivers:
                current_proposer = matches_by_receiver.get(receiver)
                if current_proposer is None or self.market.receiver_prefers(receiver, proposer, current_proposer):
                    blocking_pairs.append((proposer, receiver))

        return blocking_pairs


**Object model.** Define `MatchingPrinter`, the named objects used by the next examples.


In [ ]:
class MatchingPrinter:
    def matching(self, matches_by_proposer: dict[Person, Person]) -> str:
        return "\n".join(f"{proposer:>7} -> {receiver}" for proposer, receiver in sorted(matches_by_proposer.items()))

    def people(self, people: list[Person]) -> str:
        return ", ".join(str(person) for person in people) or "none"

    def pairs(self, pairs: list[tuple[Person, Person]]) -> str:
        return ", ".join(f"{proposer}-{receiver}" for proposer, receiver in pairs) or "none"


## 3. Create a Tiny Matching Market

Now create two groups: students and programs. Students will propose to programs in preference order.

The algorithm also works if the other side proposes. The result may change, but it will still be stable.

<details>
<summary>Hint: complete preference lists</summary>

This simple version expects everyone to rank every person on the other side.

</details>

**Example state.** Create `ari`, `bea`, `cam`, `dev`, and related helpers, the concrete values used in the next run.


In [ ]:
ari = Person("Ari")

bea = Person("Bea")

cam = Person("Cam")

dev = Person("Dev")

north = Person("North")

east = Person("East")

south = Person("South")

west = Person("West")

students = [ari, bea, cam, dev]

programs = [north, east, south, west]

student_preferences = {
    ari: [north, east, south, west],
    bea: [north, south, east, west],
    cam: [east, north, west, south],
    dev: [south, east, west, north],
}

program_preferences = {
    north: [bea, ari, cam, dev],
    east: [ari, cam, dev, bea],
    south: [dev, bea, ari, cam],
    west: [cam, dev, bea, ari],
}


**Example state.** Create `market`, the concrete values used in the next run.


In [ ]:
market = MatchingMarket(
    proposers=students,
    receivers=programs,
    proposer_preferences=student_preferences,
    receiver_preferences=program_preferences,
)

print(market.describe())


## 4. Run Gale-Shapley

The runner returns a `MatchingResult` with:

- `matches_by_proposer`: final matches
- `steps`: proposal snapshots for replay
- `blocking_pairs`: pairs that would break stability
- `is_stable`: whether the matching has no blocking pairs

<details>
<summary>Quick check</summary>

A stable matching should have `is_stable == True` and no blocking pairs.

</details>

In [3]:
runner = GaleShapleyRunner(market)
result = runner.run()
printer = MatchingPrinter()

print("Final matching:")
print(printer.matching(result.matches_by_proposer))

print("\nBlocking pairs:", printer.pairs(result.blocking_pairs))
print("Stable:", result.is_stable)

Final matching:
    Ari -> East
    Bea -> North
    Cam -> West
    Dev -> South

Blocking pairs: none
Stable: True


## 5. Replay the Proposals

The replay shows who proposes, what the receiver does, the current tentative matches, and who is still free.

<details>
<summary>Hint: why BFS or heaps are not needed here</summary>

Gale-Shapley only needs a queue of free proposers and each proposer's next untried preference. There is no shortest path or cheapest-first choice.

</details>

In [4]:
class MatchingReplay:
    def __init__(self, steps: list[ProposalStep]):
        self.steps = steps
        self.printer = MatchingPrinter()

    def show(self) -> None:
        for step in self.steps:
            print(f"Round {step.round_number}: {step.proposer} proposes to {step.receiver}")
            print(" ", step.action)
            print("  tentative matches:")
            print(self._format_receiver_matches(step.matches_by_receiver))
            print("  free proposers:", self.printer.people(step.free_proposers))
            print()

    def _format_receiver_matches(self, matches_by_receiver: dict[Person, Person]) -> str:
        rows = []
        for receiver, proposer in sorted(matches_by_receiver.items()):
            rows.append(f"    {receiver:>7} holds {proposer}")
        return "\n".join(rows) or "    none"


replay = MatchingReplay(result.steps)
replay.show()

Round 1: Ari proposes to North
  North is free, so North holds Ari.
  tentative matches:
      North holds Ari
  free proposers: Bea, Cam, Dev

Round 2: Bea proposes to North
  North switches from Ari to Bea.
  tentative matches:
      North holds Bea
  free proposers: Cam, Dev, Ari

Round 3: Cam proposes to East
  East is free, so East holds Cam.
  tentative matches:
       East holds Cam
      North holds Bea
  free proposers: Dev, Ari

Round 4: Dev proposes to South
  South is free, so South holds Dev.
  tentative matches:
       East holds Cam
      North holds Bea
      South holds Dev
  free proposers: Ari

Round 5: Ari proposes to East
  East switches from Cam to Ari.
  tentative matches:
       East holds Ari
      North holds Bea
      South holds Dev
  free proposers: Cam

Round 6: Cam proposes to North
  North rejects Cam and keeps Bea.
  tentative matches:
       East holds Ari
      North holds Bea
      South holds Dev
  free proposers: Cam

Round 7: Cam proposes to West


## 6. Your Experiments

Try changing one thing at a time:

- Swap two preferences in one list
- Let the other side propose instead
- Add a new matched pair to both sides
- Create a matching by hand and check for blocking pairs

<details>
<summary>Challenge</summary>

The small example below is designed so the proposing side changes the stable matching. Predict the difference before you run it.

</details>

**Example state.** Create `red`, `blue`, `green`, `oak`, and related helpers, the concrete values used in the next run.


In [ ]:
red = Person("Red")

blue = Person("Blue")

green = Person("Green")

oak = Person("Oak")

pine = Person("Pine")

elm = Person("Elm")

colors = [red, blue, green]

trees = [oak, pine, elm]

color_preferences = {
    red: [oak, pine, elm],
    blue: [pine, oak, elm],
    green: [oak, pine, elm],
}

tree_preferences = {
    oak: [blue, red, green],
    pine: [red, blue, green],
    elm: [red, blue, green],
}

colors_propose = MatchingMarket(colors, trees, color_preferences, tree_preferences)

trees_propose = MatchingMarket(trees, colors, tree_preferences, color_preferences)

colors_result = GaleShapleyRunner(colors_propose).run()

trees_result = GaleShapleyRunner(trees_propose).run()


**Inspect the result.** Evaluate the expression and read the output before changing parameters.


In [ ]:
print("When colors propose:")

print(MatchingPrinter().matching(colors_result.matches_by_proposer))

print("Stable:", colors_result.is_stable)

print("\nWhen trees propose:")

print(MatchingPrinter().matching(trees_result.matches_by_proposer))

print("Stable:", trees_result.is_stable)


## Visual Trace + Rigor Studio

**Problem frame.** Find a stable matching through local proposals.

**Interactive animation target.** Animate proposal arrows, rejections, and tentative matches as preferences narrow.

**Correctness handle.** A receiver's tentative match can only improve according to that receiver's preference list.

**Complexity handle.** At most n^2 proposals because each proposer asks each receiver at most once.

**Failure mode to test.** The proposer-optimal result may be receiver-pessimal, so fairness depends on who proposes.

**Studio task.** Switch the proposing side and compare both the matching and the rejected proposal history.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
